In [1]:
# ===== Kaggle: chuẩn bị dữ liệu + sinh YAML =====
from pathlib import Path
import random, textwrap

# 0) Cấu hình nhỏ
USE_EVAL1K_AS_TEST = True  # =False nếu muốn test là FishEye8K/test

# 1) Xác định đường dẫn dataset
FISHEYE_ROOT_CANDIDATES = [
    Path("/kaggle/input/fisheye8k/Fisheye8K"),  # theo panel của bạn
    Path("/kaggle/input/fisheye8k"),
    Path("/kaggle/input/Fisheye8K"),
]
DATA = next((p for p in FISHEYE_ROOT_CANDIDATES if p.exists()), None)
assert DATA is not None, "❌ Không tìm thấy FishEye8K trong /kaggle/input/*"

TR_IMG = DATA / "train/images"
TR_LBL = DATA / "train/labels"
TE_IMG = DATA / "test/images"  # test của FishEye8K 

# Eval1k (public test) theo panel của bạn:
EVAL1K_DIR = Path("/kaggle/input/eval1k/eval1k/images")

# Kiểm tra tồn tại
assert TR_IMG.exists(), f"❌ Thiếu thư mục: {TR_IMG}"
assert TR_LBL.exists(), f"❌ Thiếu thư mục: {TR_LBL}"
if USE_EVAL1K_AS_TEST:
    assert EVAL1K_DIR.exists(), f"❌ Thiếu thư mục Eval1k: {EVAL1K_DIR}"

print("✅ DATA root:", DATA)
print("✅ Eval1k images:", EVAL1K_DIR if USE_EVAL1K_AS_TEST else "(không dùng)")

# 2) Tạo split train/val từ train
img_exts = {".jpg", ".jpeg", ".png", ".bmp"}
imgs = sorted([p for p in TR_IMG.glob("*") if p.suffix.lower() in img_exts])
assert imgs, "❌ Không thấy ảnh trong train/images!"
imgs = [p for p in imgs if (TR_LBL / f"{p.stem}.txt").exists()]
assert imgs, "❌ Không thấy nhãn YOLO tương ứng trong train/labels!"

random.seed(42)
random.shuffle(imgs)
ratio = 0.20
n_val = max(1, int(len(imgs) * ratio))
val_imgs = set(imgs[:n_val])
train_imgs = [p for p in imgs if p not in val_imgs]
print(f"Split -> Train: {len(train_imgs)} | Val: {len(val_imgs)} | Total: {len(imgs)}")

# 3) Ghi file-list & YAML vào /kaggle/working
WORK = Path("/kaggle/working")
TRAIN_LIST = WORK / "train_files.txt"
VAL_LIST   = WORK / "val_files.txt"

with open(TRAIN_LIST, "w") as f:
    for p in train_imgs: f.write(str(p) + "\n")
with open(VAL_LIST, "w") as f:
    for p in sorted(val_imgs): f.write(str(p) + "\n")

test_path = EVAL1K_DIR if USE_EVAL1K_AS_TEST else TE_IMG
yaml_text = textwrap.dedent(f"""
train: {TRAIN_LIST}
val: {VAL_LIST}
test: {test_path}
nc: 5
names: [bus, bike, car, pedestrian, truck]
""").strip()

YAML_PATH = WORK / "fisheye.yaml"
YAML_PATH.write_text(yaml_text, encoding="utf-8")

print("\n📄 Đã tạo:")
print(" -", TRAIN_LIST)
print(" -", VAL_LIST)
print(" -", YAML_PATH)
print("\n--- YAML content ---\n", YAML_PATH.read_text())


✅ DATA root: /kaggle/input/fisheye8k/Fisheye8K
✅ Eval1k images: /kaggle/input/eval1k/eval1k/images
Split -> Train: 4231 | Val: 1057 | Total: 5288

📄 Đã tạo:
 - /kaggle/working/train_files.txt
 - /kaggle/working/val_files.txt
 - /kaggle/working/fisheye.yaml

--- YAML content ---
 train: /kaggle/working/train_files.txt
val: /kaggle/working/val_files.txt
test: /kaggle/input/eval1k/eval1k/images
nc: 5
names: [bus, bike, car, pedestrian, truck]


In [2]:
!pip install ultralytics --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 14.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [3]:
from ultralytics import YOLO
import torch

# 1️⃣ Chọn device
device = 0 if torch.cuda.is_available() else "cpu"
print("🚀 Device:", torch.cuda.get_device_name(0) if device == 0 else "CPU")

# 2️⃣ Chọn backbone phù hợp
# - yolov8m.pt: cân bằng speed/accuracy → phù hợp T4
# - yolov8l.pt: nếu được cấp V100/A100, accuracy cao hơn
# - yolov8s.pt: train nhanh để thử
model = YOLO("yolov8m.pt")

# 3️⃣ Train model
results = model.train(
    data="/kaggle/working/fisheye.yaml",  # YAML file bạn đã sinh
    epochs=80,              # tăng lên để mô hình hội tụ tốt hơn
    imgsz=960,              # lớn hơn 640 → detect object nhỏ tốt hơn
    batch=16,               # Kaggle T4 chịu được 16 cho yolov8m; V100 có thể 32
    device=device,
    freeze=10, 
    # 🔸 Data augmentation - khuyến nghị từ bài báo bạn gửi
    mosaic=0.8,             # giữ mosaic nhưng giảm nhẹ để tránh oversynth
    mixup=0.15,            # thêm mixup nhẹ giúp regularize
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.4, shear=2.0,
    perspective=0.0005,     # fisheye biến dạng nhẹ
    fliplr=0.5, flipud=0.0,

    # 🔸 Optimization
    lr0=0.01, lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    cos_lr=True,           # cosine decay giúp hội tụ mượt hơn

    # 🔸 Early stopping & mixed precision
    patience=20,          # cho phép train đủ lâu nếu val plateau chậm
    amp=True,             # mixed precision → tăng tốc & tiết kiệm VRAM

    # 🔸 Logging & Eval
    save_json=True,       # để xuất COCO JSON tính mAP/F1
    plots=True,
    verbose=True,
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Device: Tesla P100-PCIE-16GB
Ultralytics 8.3.204 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/fisheye.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, i

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1057      22225      0.931      0.897      0.953      0.674
                   bus        309        390      0.934      0.938      0.971      0.749
                  bike       1045      12394      0.936      0.904      0.966      0.632
                   car       1049       7281      0.951      0.947      0.984      0.733
            pedestrian        693       1733      0.899      0.814      0.897      0.534
                 truck        279        427      0.937      0.881      0.945      0.724
Speed: 0.4ms preprocess, 20.0ms inference, 0.0ms loss, 1.8ms postprocess per image
Saving /kaggle/working/runs/detect/train/predictions.json...
Results saved to /kaggle/working/runs/detect/train


In [4]:
# === SAVE TRAINED WEIGHTS TO /kaggle/working ===
from pathlib import Path
import shutil

runs_root = Path("runs/detect")
assert runs_root.exists(), "❌ Chưa có runs/detect (hãy train xong rồi chạy cell này)."

latest_run = max(runs_root.glob("*/"), key=lambda p: p.stat().st_mtime)
wdir = latest_run / "weights"
best_pt = wdir / "best.pt"
last_pt = wdir / "last.pt"
assert best_pt.exists(), f"❌ Không thấy {best_pt}"

# copy ra working dir
out_best = Path("/kaggle/working/best.pt")
out_last = Path("/kaggle/working/last.pt")
shutil.copy(best_pt, out_best)
if last_pt.exists():
    shutil.copy(last_pt, out_last)

print("✅ Saved model:", out_best, "|", out_last if out_last.exists() else "(no last.pt)")


✅ Saved model: /kaggle/working/best.pt | /kaggle/working/last.pt


In [5]:
from IPython.display import FileLink
FileLink('/kaggle/working/best.pt')


/kaggle/working/best.pt

In [6]:
from ultralytics import YOLO
import torch, os, json
from pathlib import Path

MODEL_PATH = "/kaggle/working/best.pt"
EVAL1K_DIR = "/kaggle/input/eval1k/eval1k/images"
OUT_JSON   = "/kaggle/working/eval_predictions.json"

assert Path(MODEL_PATH).exists(), "❌ Thiếu best.pt"
assert Path(EVAL1K_DIR).exists(), "❌ Thiếu Eval1k/images"

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL_PATH)

# predict
results = model.predict(
    source=EVAL1K_DIR,
    conf=0.25,
    iou = 0.5,
    imgsz=640,
    device=device,
    save=False,
    save_txt=False,
    save_conf=True,
    stream=True
)

# mapping filename -> image_id
def get_image_Id(fname: str) -> int:
    name = Path(fname).stem
    cam, scene, frame = name.split('_')
    scene_idx = ['M','A','E','N'].index(scene)
    cam_idx   = int(cam.replace("camera",""))
    frame_idx = int(frame)
    return int(f"{cam_idx}{scene_idx}{frame_idx}")

# build predictions
output = []
for r in results:
    image_id = get_image_Id(os.path.basename(r.path))
    for b in r.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        output.append({
            "image_id": image_id,
            "category_id": int(b.cls[0].item()),
            "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
            "score": float(b.conf[0].item())
        })

with open(OUT_JSON, "w") as f:
    json.dump(output, f)

print(f"✅ JSON saved: {OUT_JSON} | #boxes: {len(output)}")



image 1/1000 /kaggle/input/eval1k/eval1k/images/camera19_A_0.png: 640x640 9 bikes, 15 cars, 1 pedestrian, 15.0ms
image 2/1000 /kaggle/input/eval1k/eval1k/images/camera19_A_1.png: 640x640 14 bikes, 2 cars, 1 pedestrian, 1 truck, 14.9ms
image 3/1000 /kaggle/input/eval1k/eval1k/images/camera19_A_10.png: 640x640 2 buss, 24 bikes, 11 cars, 3 pedestrians, 15.1ms
image 4/1000 /kaggle/input/eval1k/eval1k/images/camera19_A_11.png: 640x640 1 bus, 19 bikes, 18 cars, 14.9ms
image 5/1000 /kaggle/input/eval1k/eval1k/images/camera19_A_12.png: 640x640 3 buss, 12 bikes, 9 cars, 1 pedestrian, 15.0ms
image 6/1000 /kaggle/input/eval1k/eval1k/images/camera19_A_13.png: 640x640 20 bikes, 6 cars, 15.0ms
image 7/1000 /kaggle/input/eval1k/eval1k/images/camera19_A_14.png: 640x640 1 bus, 17 bikes, 10 cars, 2 pedestrians, 1 truck, 14.9ms
image 8/1000 /kaggle/input/eval1k/eval1k/images/camera19_A_15.png: 640x640 2 buss, 11 bikes, 19 cars, 15.0ms
image 9/1000 /kaggle/input/eval1k/eval1k/images/camera19_A_16.png: 64

In [7]:
from IPython.display import FileLink
FileLink('/kaggle/working/eval_predictions.json')


/kaggle/working/eval_predictions.json

In [8]:
# === Evaluate lại để lấy thêm F1 ===
metrics = model.val(
    data=YAML_PATH,   # file yaml bạn tạo
    conf=0.25,
    iou=0.5
)

# Lấy P và R
P = metrics.results_dict["metrics/precision(B)"]
R = metrics.results_dict["metrics/recall(B)"]

# Tính F1 theo công thức
F1 = 2 * (P * R) / (P + R + 1e-6)

print("Precision:", P)
print("Recall:", R)
print("F1-score:", F1)
print("mAP50:", metrics.results_dict["metrics/mAP50(B)"])
print("mAP50-95:", metrics.results_dict["metrics/mAP50-95(B)"])


Ultralytics 8.3.204 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
val: Fast image access ✅ (ping: 0.9±1.0 ms, read: 2544.1±956.2 MB/s, size: 3072.3 KB)
val: Scanning /kaggle/input/fisheye8k/Fisheye8K/train/labels... 1057 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1057/1057 150.2it/s 7.0s
val: /kaggle/input/fisheye8k/Fisheye8K/train/images/camera17_A_57.png: 1 duplicate labels removed
val: /kaggle/input/fisheye8k/Fisheye8K/train/images/camera17_A_67.png: 1 duplicate labels removed
val: /kaggle/input/fisheye8k/Fisheye8K/train/images/camera17_A_70.png: 1 duplicate labels removed
val: /kaggle/input/fisheye8k/Fisheye8K/train/images/camera17_A_85.png: 1 duplicate labels removed
WARNING ⚠️ val: Cache directory /kaggle/input/fisheye8k/Fisheye8K/train is not writeable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.2it/s 30.8s


/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1057      22225      0.931      0.899      0.949      0.705
                   bus        309        390      0.934      0.941      0.969      0.775
                  bike       1045      12394      0.936      0.908       0.96      0.664
                   car       1049       7281       0.95       0.95      0.977      0.755
            pedestrian        693       1733      0.903      0.815        0.9      0.579
                 truck        279        427      0.934      0.881       0.94      0.753
Speed: 1.0ms preprocess, 19.7ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
Precision: 0.9313892816401248
Recall: 0.8988545129038661
F1-score: 0.9148322252500678
mAP50: 0.9491689745884584
mAP50-95: 0.705089836107726
